<a href="https://colab.research.google.com/github/apar-tech/rag-chatbot/blob/main/phase2/ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    !pip install -q \
    chromadb \
    wikipedia-api \
    langchain-text-splitters \
    google-genai
    print("✓ Libraries installed successfully!")
except subprocess.CalledProcessError as e:
    print(f"✗ Installation Error (subprocess): {e}")
    print("  Check your internet connection and try again")
except ImportError as e:
    print(f"✗ Import Error during installation: {e}")
except Exception as e:
    print(f"✗ Unexpected Installation Error: {type(e).__name__}: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 568.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [2]:
try:
    import os
    import time
    import pickle
    import subprocess
    import wikipediaapi
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from google import genai
    from google.colab import userdata
    import chromadb
    print("✓ All libraries imported successfully!")
except ImportError as e:
    print(f"✗ Import Error: {e}")
    print("  Please ensure all packages are installed correctly")
except Exception as e:
    print(f"✗ Unexpected Import Error: {type(e).__name__}: {e}")

✓ All libraries imported successfully!


Download 3 files from wikipedia Artificial intelligence,Machine learning,Deep learning

In [3]:
try:
    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="networking-rag-project"
    )
    print("✓ Wikipedia API client initialized")
except Exception as e:
    print(f"✗ Error initializing Wikipedia API: {type(e).__name__}: {e}")
    wiki = None

try:
    if wiki is None:
        raise ValueError("Wikipedia API client not initialized")

    topics = [
        "Artificial intelligence",
        "Machine learning",
        "Deep learning"
    ]

    # Create directory with exception handling
    try:
        os.makedirs("digital_docs", exist_ok=True)
        print("✓ Created directory: digital_docs")
    except PermissionError as e:
        print(f"✗ Permission Error creating directory: {e}")
    except OSError as e:
        print(f"✗ OS Error creating directory: {e}")

    downloaded_count = 0

    for topic in topics:
        try:
            page = wiki.page(topic)

            if page.exists():
                filename = topic.replace("/", "_").replace(" ", "_") + ".txt"

                try:
                    with open(f"digital_docs/{filename}", "w", encoding="utf-8") as f:
                        f.write(page.text)
                    print(f"✓ Saved: {filename}")
                    downloaded_count += 1
                except IOError as e:
                    print(f"✗ File write error for {topic}: {e}")
                except UnicodeEncodeError as e:
                    print(f"✗ Unicode encoding error for {topic}: {e}")
            else:
                print(f"✗ Page not found: {topic}")

        except wikipediaapi.exceptions.WikipediaException as e:
            print(f"✗ Wikipedia API error for {topic}: {e}")
        except Exception as e:
            print(f"✗ Unexpected error downloading {topic}: {type(e).__name__}: {e}")

    print(f"\n✓ Downloaded {downloaded_count} out of {len(topics)} documents successfully!")

except ValueError as e:
    print(f"✗ Value Error: {e}")
except Exception as e:
    print(f"✗ Unexpected Download Error: {type(e).__name__}: {e}")

✓ Wikipedia API client initialized
✓ Created directory: digital_docs
✓ Saved: Artificial_intelligence.txt
✓ Saved: Machine_learning.txt
✓ Saved: Deep_learning.txt

✓ Downloaded 3 out of 3 documents successfully!


Load files,splits into chunks of 2500 words,embeded each chunk using Gemini,and save to chromaDB

In [5]:
try:
    documents = []
    folder_path = "digital_docs"

    # Check if directory exists
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Directory '{folder_path}' not found")

    # Check if directory is empty
    files = os.listdir(folder_path)
    if not files:
        raise ValueError(f"No files found in '{folder_path}'")

    txt_files = [f for f in files if f.endswith(".txt")]
    if not txt_files:
        raise ValueError(f"No .txt files found in '{folder_path}'")

    for file in txt_files:
        file_path = os.path.join(folder_path, file)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                if content.strip():  # Check if file is not empty
                    documents.append(content)
                    print(f"✓ Loaded: {file} ({len(content)} characters)")
                else:
                    print(f"✗ Warning: {file} is empty")
        except FileNotFoundError as e:
            print(f"✗ File not found: {file_path}")
        except PermissionError as e:
            print(f"✗ Permission denied: {file_path}")
        except UnicodeDecodeError as e:
            print(f"✗ Unicode decode error for {file}: {e}")
        except IOError as e:
            print(f"✗ IO Error reading {file}: {e}")

    if documents:
        print(f"\n✓ Documents loaded successfully!")
        print(f"  Number of documents: {len(documents)}")
        print(f"  Total characters: {sum(len(doc) for doc in documents)}")
    else:
        raise ValueError("No documents were successfully loaded")

except FileNotFoundError as e:
    print(f"✗ FileNotFound Error: {e}")
    print("  Please run the download cell first")
except ValueError as e:
    print(f"✗ Value Error: {e}")
except Exception as e:
    print(f"✗ Unexpected Loading Error: {type(e).__name__}: {e}")

✓ Loaded: Deep_learning.txt (55823 characters)
✓ Loaded: Machine_learning.txt (58114 characters)
✓ Loaded: Artificial_intelligence.txt (84893 characters)

✓ Documents loaded successfully!
  Number of documents: 3
  Total characters: 198830


In [6]:
#  Chunk documents with exception handling
try:
    if 'documents' not in locals() or not documents:
        raise ValueError("No documents available for chunking")

    try:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=2500,
            chunk_overlap=50
        )
        print("✓ Text splitter initialized")
    except Exception as e:
        print(f"✗ Error initializing text splitter: {e}")
        raise

    chunks = []
    chunk_errors = 0

    for i, document in enumerate(documents):
        try:
            document_chunks = splitter.split_text(document)
            chunks.extend(document_chunks)
            print(f"✓ Document {i+1}: Created {len(document_chunks)} chunks")
        except TypeError as e:
            print(f"✗ Type error in document {i+1}: {e}")
            chunk_errors += 1
        except ValueError as e:
            print(f"✗ Value error in document {i+1}: {e}")
            chunk_errors += 1
        except Exception as e:
            print(f"✗ Unexpected error in document {i+1}: {type(e).__name__}: {e}")
            chunk_errors += 1

    if chunks:
        print(f"\n✓ Chunking completed successfully!")
        print(f"  Total Chunks: {len(chunks)}")
        if chunk_errors > 0:
            print(f"  ⚠️ Errors encountered: {chunk_errors}")
    else:
        raise ValueError("No chunks were created")

except NameError as e:
    print(f"✗ NameError - 'documents' not defined: {e}")
    print("  Please run the document loading cell first")
except ValueError as e:
    print(f"✗ Value Error: {e}")
except Exception as e:
    print(f"✗ Unexpected Chunking Error: {type(e).__name__}: {e}")

✓ Text splitter initialized
✓ Document 1: Created 31 chunks
✓ Document 2: Created 34 chunks
✓ Document 3: Created 46 chunks

✓ Chunking completed successfully!
  Total Chunks: 111


In [8]:
# Generate embeddings with rate limiting and comprehensive exception handling
try:
    if 'chunks' not in locals() or not chunks:
        raise ValueError("No chunks available for embedding")

    # Initialize Gemini client
    try:
        client = genai.Client(api_key=userdata.get("geminikey"))
        print("✓ Gemini client initialized")
    except KeyError as e:
        print(f"✗ API Key Error: {e}")
        print("  Please set up geminikey in Colab secrets")
        raise
    except Exception as e:
        print(f"✗ Error initializing Gemini client: {type(e).__name__}: {e}")
        raise

    chunk_embeddings = []
    embedding_errors = []
    rate_limit_delay = 2  # seconds between requests

    print(f"Starting embedding generation for {len(chunks)} chunks...")
    print(f"Rate limit: {rate_limit_delay} second(s) between requests")

    for i, chunk in enumerate(chunks):
        try:
            # Add delay for rate limiting
            if i > 0:
                time.sleep(rate_limit_delay)

            # Validate chunk
            if not chunk or not chunk.strip():
                print(f"⚠️ Warning: Empty chunk at index {i}, skipping...")
                continue

            # Generate embedding
            try:
                response = client.models.embed_content(
                    model="models/gemini-embedding-001",
                    contents=chunk
                )

                # Validate response
                if not hasattr(response, 'embeddings'):
                    raise AttributeError("Response missing 'embeddings' attribute")
                if not response.embeddings:
                    raise ValueError("Empty embeddings list in response")
                if not hasattr(response.embeddings[0], 'values'):
                    raise AttributeError("Embedding missing 'values' attribute")

                chunk_embeddings.append(response.embeddings[0].values)

                # Progress update
                if (i + 1) % 10 == 0:
                    print(f"✓ Processed {i + 1}/{len(chunks)} chunks")

            except AttributeError as e:
                print(f"✗ Attribute Error for chunk {i}: {e}")
                embedding_errors.append((i, str(e)))
                continue
            except ValueError as e:
                print(f"✗ Value Error for chunk {i}: {e}")
                embedding_errors.append((i, str(e)))
                continue
            except TypeError as e:
                print(f"✗ Type Error for chunk {i}: {e}")
                embedding_errors.append((i, str(e)))
                continue

        except Exception as e:
            print(f"✗ Unexpected error for chunk {i}: {type(e).__name__}: {e}")
            embedding_errors.append((i, str(e)))
            continue

    # Results summary
    if chunk_embeddings:
        print(f"\n✓ Embedding generation completed!")
        print(f"  Successful embeddings: {len(chunk_embeddings)}/{len(chunks)}")
        print(f"  Failed embeddings: {len(embedding_errors)}")
        if embedding_errors:
            print(f"  Errors at chunks: {[idx for idx, _ in embedding_errors[:10]]}...")
    else:
        raise ValueError("No embeddings were generated successfully")

except NameError as e:
    print(f"✗ NameError - Required variable not defined: {e}")
    print(f"  'chunks' defined: {'chunks' in locals()}")
except ValueError as e:
    print(f"✗ Value Error: {e}")
except Exception as e:
    print(f"✗ Unexpected Embedding Error: {type(e).__name__}: {e}")

✓ Gemini client initialized
Starting embedding generation for 111 chunks...
Rate limit: 2 second(s) between requests
✓ Processed 10/111 chunks
✓ Processed 20/111 chunks
✓ Processed 30/111 chunks
✓ Processed 40/111 chunks
✓ Processed 50/111 chunks
✓ Processed 60/111 chunks
✓ Processed 70/111 chunks
✓ Processed 80/111 chunks
✓ Processed 90/111 chunks
✓ Processed 100/111 chunks
✓ Processed 110/111 chunks

✓ Embedding generation completed!
  Successful embeddings: 111/111
  Failed embeddings: 0


In [9]:
# Store in ChromaDB with exception handling
try:
    if 'chunks' not in locals() or not chunks:
        raise ValueError("No chunks available for storage")
    if 'chunk_embeddings' not in locals() or not chunk_embeddings:
        raise ValueError("No embeddings available for storage")

    # Validate lengths match
    if len(chunks) != len(chunk_embeddings):
        raise ValueError(
            f"Length mismatch: chunks={len(chunks)}, embeddings={len(chunk_embeddings)}"
        )

    try:
        # Initialize ChromaDB client
        chroma_client = chromadb.PersistentClient(path="./digital_chromadb")
        print("✓ ChromaDB client initialized")
    except PermissionError as e:
        print(f"✗ Permission Error creating ChromaDB directory: {e}")
        raise
    except Exception as e:
        print(f"✗ Error initializing ChromaDB: {type(e).__name__}: {e}")
        raise

    try:
        # Get or create collection
        collection = chroma_client.get_or_create_collection(name="digital_docs")
        print(f"✓ Collection 'digital_docs' ready (existing records: {collection.count()})")
    except Exception as e:
        print(f"✗ Error accessing collection: {type(e).__name__}: {e}")
        raise

    try:
        # Prepare data for insertion
        ids = [f"chunk_{i}" for i in range(len(chunks))]

        # Add to collection with error handling for each batch
        batch_size = 50
        total_added = 0

        for i in range(0, len(chunks), batch_size):
            batch_end = min(i + batch_size, len(chunks))
            try:
                collection.add(
                    ids=ids[i:batch_end],
                    documents=chunks[i:batch_end],
                    embeddings=chunk_embeddings[i:batch_end]
                )
                total_added += (batch_end - i)
                print(f"✓ Added batch {i//batch_size + 1}: records {i+1}-{batch_end}")
            except ValueError as e:
                print(f"✗ Value Error in batch {i//batch_size + 1}: {e}")
                # Try individual inserts for failed batch
                for j in range(i, batch_end):
                    try:
                        collection.add(
                            ids=[ids[j]],
                            documents=[chunks[j]],
                            embeddings=[chunk_embeddings[j]]
                        )
                        total_added += 1
                        print(f"  ✓ Added individual chunk {j}")
                    except Exception as e2:
                        print(f"  ✗ Failed to add chunk {j}: {e2}")
            except Exception as e:
                print(f"✗ Error in batch {i//batch_size + 1}: {type(e).__name__}: {e}")

        print(f"\n✓ ChromaDB storage completed!")
        print(f"  Total records added: {total_added}")
        print(f"  Total records in collection: {collection.count()}")

    except Exception as e:
        print(f"✗ Error adding to ChromaDB: {type(e).__name__}: {e}")
        raise

except NameError as e:
    print(f"✗ NameError - Required variable not defined: {e}")
except ValueError as e:
    print(f"✗ Value Error: {e}")
except Exception as e:
    print(f"✗ Unexpected ChromaDB Error: {type(e).__name__}: {e}")

✓ ChromaDB client initialized
✓ Collection 'digital_docs' ready (existing records: 0)
✓ Added batch 1: records 1-50
✓ Added batch 2: records 51-100
✓ Added batch 3: records 101-111

✓ ChromaDB storage completed!
  Total records added: 111
  Total records in collection: 111


In [10]:
 # Verify ChromaDB storage with exception handling
try:
    # Check if collection exists in locals
    if 'collection' not in locals():
        raise NameError("collection variable not defined")

    # Get collection info
    try:
        collection_name = collection.name
        collection_count = collection.count()
        print("✓ ChromaDB verification successful!")
        print(f"  Collection Name: {collection_name}")
        print(f"  Total Records: {collection_count}")

        # Optional: Check if collection is accessible
        if collection_count > 0:
            # Try to peek at first record
            try:
                sample = collection.get(limit=1)
                if sample and sample['ids']:
                    print(f"  Sample ID: {sample['ids'][0]}")
                    print(f"  Sample document length: {len(sample['documents'][0]) if sample['documents'] else 0}")
            except Exception as e:
                print(f"  ⚠️ Could not peek at sample: {e}")
        else:
            print("  ⚠️ Collection is empty")

    except AttributeError as e:
        print(f"✗ Attribute Error - collection may not be properly initialized: {e}")
    except ValueError as e:
        print(f"✗ Value Error - invalid collection state: {e}")
    except Exception as e:
        print(f"✗ Error accessing collection: {type(e).__name__}: {e}")

except NameError as e:
    print(f"✗ NameError - {e}")
    print("  Please run the ChromaDB storage cell first")
except Exception as e:
    print(f"✗ Unexpected Verification Error: {type(e).__name__}: {e}")

✓ ChromaDB verification successful!
  Collection Name: digital_docs
  Total Records: 111
  Sample ID: chunk_0
  Sample document length: 1428


Pickle is a Python module used to save the python objects to a file and load them back later.Instead of creating an object every time the program runs,you can store it and reuse

In [11]:
#  Save to pickle files with exception handling
try:
    if 'chunks' not in locals() or not chunks:
        raise ValueError("No chunks available to save")
    if 'chunk_embeddings' not in locals() or not chunk_embeddings:
        raise ValueError("No embeddings available to save")

    # Save chunks
    try:
        with open("chunks.pkl", "wb") as f:
            pickle.dump(chunks, f)
        print(f"✓ chunks.pkl saved successfully ({len(chunks)} items)")
    except TypeError as e:
        print(f"✗ Type Error saving chunks.pkl: {e}")
        print("  Check if chunks contains serializable data")
    except PermissionError as e:
        print(f"✗ Permission Error saving chunks.pkl: {e}")
    except IOError as e:
        print(f"✗ IO Error saving chunks.pkl: {e}")
    except Exception as e:
        print(f"✗ Unexpected error saving chunks.pkl: {type(e).__name__}: {e}")

    # Save embeddings
    try:
        with open("chunk_embeddings.pkl", "wb") as f:
            pickle.dump(chunk_embeddings, f)
        print(f"✓ chunk_embeddings.pkl saved successfully ({len(chunk_embeddings)} items)")
    except TypeError as e:
        print(f"✗ Type Error saving chunk_embeddings.pkl: {e}")
        print("  Check if chunk_embeddings contains serializable data")
    except PermissionError as e:
        print(f"✗ Permission Error saving chunk_embeddings.pkl: {e}")
    except IOError as e:
        print(f"✗ IO Error saving chunk_embeddings.pkl: {e}")
    except Exception as e:
        print(f"✗ Unexpected error saving chunk_embeddings.pkl: {type(e).__name__}: {e}")

    # Verify files were created
    try:
        files_created = []
        for filename in ["chunks.pkl", "chunk_embeddings.pkl"]:
            if os.path.exists(filename):
                file_size = os.path.getsize(filename)
                files_created.append(f"{filename} ({file_size} bytes)")

        if files_created:
            print(f"\n✓ Files saved successfully!")
            print(f"  Created: {', '.join(files_created)}")
        else:
            raise FileNotFoundError("No pickle files were created")

    except FileNotFoundError as e:
        print(f"✗ File not found after saving: {e}")
    except OSError as e:
        print(f"✗ OS Error checking files: {e}")

except NameError as e:
    print(f"✗ NameError - Required variable not defined: {e}")
except ValueError as e:
    print(f"✗ Value Error: {e}")
except Exception as e:
    print(f"✗ Unexpected Pickle Error: {type(e).__name__}: {e}")

✓ chunks.pkl saved successfully (111 items)
✓ chunk_embeddings.pkl saved successfully (111 items)

✓ Files saved successfully!
  Created: chunks.pkl (199427 bytes), chunk_embeddings.pkl (3070468 bytes)


In [12]:
# Final summary with exception handling
try:
    print("=" * 60)
    print("✓✓✓ RAG PIPELINE COMPLETED SUCCESSFULLY ✓✓✓")
    print("=" * 60)

    # Check all components
    components = {
        "Documents": len(documents) if 'documents' in locals() else 0,
        "Chunks": len(chunks) if 'chunks' in locals() else 0,
        "Embeddings": len(chunk_embeddings) if 'chunk_embeddings' in locals() else 0,
        "ChromaDB Records": collection.count() if 'collection' in locals() else 0
    }

    for name, value in components.items():
        status = "✓" if value > 0 else "✗"
        print(f"{status} {name}: {value}")

    print("=" * 60)

    if all(value > 0 for value in components.values()):
        print("✓ All components are ready for RAG operations!")
    else:
        print("⚠️ Some components are missing. Check error messages above.")

except NameError as e:
    print(f"✗ NameError - Some variables not defined: {e}")
except Exception as e:
    print(f"✗ Unexpected Summary Error: {type(e).__name__}: {e}")

✓✓✓ RAG PIPELINE COMPLETED SUCCESSFULLY ✓✓✓
✓ Documents: 3
✓ Chunks: 111
✓ Embeddings: 111
✓ ChromaDB Records: 111
✓ All components are ready for RAG operations!


**Rate Limiting**: Added 2-second delay between embedding API calls to avoid hitting rate limits

**Batch Processing**: ChromaDB insertion is done in batches of 50 for better performance

**Individual Exception Handling**: Each operation has its own try-except block with specific error types

**Data Validation**: Checks for empty data, format mismatches, and type errors

**Progress Tracking**: Shows progress for embedding generation and ChromaDB insertion

**Error Recovery**: Attempts to handle individual failures without stopping the entire process

**Comprehensive Checks**: Validates variable existence and data consistency at each step

